In [140]:
import json
import torch
import pandas as pd
import kagglehub
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn

In [141]:
# Set device (GPU support for Mac M-series or NVIDIA)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [142]:
def initialize_model(model_name, num_classes=2, weights='DEFAULT'):
    """Build model architecture; use weights='DEFAULT' or None."""
    if model_name == 'resnet50':
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == 'vit':
        model = models.vit_b_16(weights=weights)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    else:
        raise ValueError(f'Unknown model_name: {model_name}')
    return model.to(device)

In [143]:
import json
import os
import torch
from collections import OrderedDict

def load_results(json_path, models_dir):
    with open(json_path, 'r') as f:
        metadata = json.load(f)

    results = {}

    for model_name, data in metadata.items():
        print(f"Restoring {model_name}...")
        model_path = os.path.join(models_dir, f"{model_name}_best.pth")
        model_obj = None
        
        if os.path.exists(model_path):
            loaded = torch.load(model_path, map_location=device)
            if isinstance(loaded, torch.nn.Module):
                model_obj = loaded.to(device)
            else:
                sd = loaded.get('state_dict', loaded) if isinstance(loaded, dict) else loaded
                sd = OrderedDict((k.replace('module.', ''), v) for k, v in dict(sd).items())
                model = initialize_model(model_name, num_classes=data.get('num_classes', 2), weights=None)
                model.load_state_dict(sd)
                model_obj = model.to(device)
        else:
            print(f"Warning: Model file not found at {model_path}")
        results[model_name] = {
            'model': model_obj,
            'history': data.get('history', []),
            'training_time': data.get('training_time', 0.0)
        }
    return results

# Usage
results = load_results('results_metadata.json', 'models1')

for name, data in results.items():
    if data['model']:
        print(f"Model: {name} | Training Time: {data['training_time']:.2f}s | Final Acc: {data['history'][-1] if data['history'] else 'N/A'}")


Restoring resnet50...
Restoring efficientnet_b0...
Restoring vit...
Model: resnet50 | Training Time: 1745.43s | Final Acc: 0.9745
Model: efficientnet_b0 | Training Time: 1429.13s | Final Acc: 0.9809500000000001
Model: vit | Training Time: 5898.07s | Final Acc: 0.9509000000000001


In [139]:
# Configuration
CONFIG = {
    "BATCH_SIZE": 32,
    "IMG_SIZE": 224,
    # Set to 0 for notebooks to avoid multiprocessing import issues
    "NUM_WORKERS": 0
}

In [150]:
# --- Download & Setup Function (restore original behavior) ---
def setup_external_datasets():
    print("Downloading external datasets via KaggleHub...")
    datasets_info = {}

    try:
        print("\n1. Downloading CashBowman (General AI)...")
        path_cb = kagglehub.dataset_download("cashbowman/ai-generated-images-vs-real-images")
        datasets_info["General_AI_Older"] = path_cb
    except Exception as e:
        print(f"Could not download CashBowman: {e}")

    try:
        print("\n2. Downloading Midjourney v6 (Specific)...")
        path_mj = kagglehub.dataset_download("mariammarioma/midjourney-cifake-inspired")
        datasets_info["Midjourney_v6"] = os.path.join(path_mj, "Midjourney", "test")
    except Exception as e:
        print(f"Could not download Midjourney data: {e}")

    try:
        print("\n3. Downloading DALL-E 3 (Specific)...")
        path_de = kagglehub.dataset_download("sourceduty/chatgpt-dall-e-3-images-and-sliced-gifs")
        datasets_info["DALLE_3"] = os.path.join(path_de)
    except Exception as e:
        print(f"Could not download DALL-E 3 data: {e}")

    return datasets_info


# --- EXECUTE ---
# 1. Download Data
external_data_map = setup_external_datasets()
external_data_map



1. Downloading CashBowman (General AI)...

2. Downloading Midjourney v6 (Specific)...

3. Downloading DALL-E 3 (Specific)...


{'General_AI_Older': '/Users/souhardyasaha/.cache/kagglehub/datasets/cashbowman/ai-generated-images-vs-real-images/versions/1',
 'Midjourney_v6': '/Users/souhardyasaha/.cache/kagglehub/datasets/mariammarioma/midjourney-cifake-inspired/versions/1/Midjourney/test',
 'DALLE_3': '/Users/souhardyasaha/.cache/kagglehub/datasets/sourceduty/chatgpt-dall-e-3-images-and-sliced-gifs/versions/3'}

In [151]:
import os
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 1. Define the specific paths based on your screenshot structure
base_root = external_data_map.get("General_AI_Older", "")

# Note: Based on the screenshot, images are inside the second nested folder
real_art_path = os.path.join(base_root, 'RealArt', 'RealArt')
ai_art_path = os.path.join(base_root, 'AiArtData', 'AiArtData')

# 2. Custom Dataset to handle the specific folder structure
class CashBowmanBenchmarkDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Load Real Images (Label 0)
        # Check if dir exists to avoid errors
        if os.path.exists(real_dir):
            for img_name in os.listdir(real_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    self.image_paths.append(os.path.join(real_dir, img_name))
                    self.labels.append(0) # Assuming 0 is Real
        
        # Load AI Images (Label 1)
        if os.path.exists(fake_dir):
            for img_name in os.listdir(fake_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    self.image_paths.append(os.path.join(fake_dir, img_name))
                    self.labels.append(1) # Assuming 1 is Fake/AI

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert("RGBA").convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a blank tensor and label 0 to prevent crash, or handle differently
            return torch.zeros((3, 224, 224)), 0

# 3. Define Transforms (Standard for Pre-trained models)
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. Prepare DataLoader
benchmark_dataset = CashBowmanBenchmarkDataset(real_art_path, ai_art_path, transform=eval_transform)
benchmark_loader = DataLoader(benchmark_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Dataset Loaded. Total Images: {len(benchmark_dataset)}")
print(f"Real Path: {real_art_path}")
print(f"AI Path: {ai_art_path}")
print("-" * 30)

# 5. Evaluation Loop
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

def benchmark_model(model, dataloader, model_name):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total if total > 0 else 0
    return accuracy

# 6. Run Benchmark on loaded models
# 'results' dict comes from your previous cell
print(f"{'Model':<20} | {'CashBowman Accuracy':<20}")
print("-" * 45)

for model_name, data in results.items():
    if data['model']:
        acc = benchmark_model(data['model'], benchmark_loader, model_name)
        print(f"{model_name:<20} | {acc:.2f}%")
    else:
        print(f"{model_name:<20} | Model not loaded")

Dataset Loaded. Total Images: 970
Real Path: /Users/souhardyasaha/.cache/kagglehub/datasets/cashbowman/ai-generated-images-vs-real-images/versions/1/RealArt/RealArt
AI Path: /Users/souhardyasaha/.cache/kagglehub/datasets/cashbowman/ai-generated-images-vs-real-images/versions/1/AiArtData/AiArtData
------------------------------
Model                | CashBowman Accuracy 
---------------------------------------------
resnet50             | 55.36%
efficientnet_b0      | 55.26%
vit                  | 55.36%


In [152]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 1. Define the specific path you provided
# This path is treated as the root containing 'REAL' and 'FAKE' folders
test_dir = external_data_map.get("Midjourney_v6", "")

print(f"Targeting Dataset Directory: {test_dir}")

# 2. Define Dataset Class
class MidjourneyBenchmarkDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Define paths for REAL and FAKE inside the test directory
        real_dir = os.path.join(root_dir, 'REAL')
        fake_dir = os.path.join(root_dir, 'FAKE')
        
        # Load Real Images (Label 0)
        if os.path.exists(real_dir):
            for img_name in os.listdir(real_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    self.image_paths.append(os.path.join(real_dir, img_name))
                    self.labels.append(0) # 0 = Real
        else:
            print(f"Warning: REAL folder not found at {real_dir}")

        # Load Fake Images (Label 1)
        if os.path.exists(fake_dir):
            for img_name in os.listdir(fake_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    self.image_paths.append(os.path.join(fake_dir, img_name))
                    self.labels.append(1) # 1 = Fake
        else:
            print(f"Warning: FAKE folder not found at {fake_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            # FIX: Convert to RGBA first to handle transparency, then to RGB
            image = Image.open(img_path).convert("RGBA").convert("RGB")
            
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros((3, 224, 224)), 0

# 3. Setup DataLoader
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize Dataset and Loader
mj_dataset = MidjourneyBenchmarkDataset(test_dir, transform=eval_transform)
mj_loader = DataLoader(mj_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Midjourney Dataset Loaded. Total Images: {len(mj_dataset)}")

# 4. Evaluation Function
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

def benchmark_model(model, dataloader, model_name):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total if total > 0 else 0
    return accuracy

# 5. Run Benchmark
# Uses the 'results' dictionary from your previous cells
print(f"\n{'Model':<20} | {'Midjourney Accuracy':<20}")
print("-" * 45)

for model_name, data in results.items():
    if data['model']:
        acc = benchmark_model(data['model'], mj_loader, model_name)
        print(f"{model_name:<20} | {acc:.2f}%")
    else:
        print(f"{model_name:<20} | Model not loaded")

Targeting Dataset Directory: /Users/souhardyasaha/.cache/kagglehub/datasets/mariammarioma/midjourney-cifake-inspired/versions/1/Midjourney/test
Midjourney Dataset Loaded. Total Images: 1000

Model                | Midjourney Accuracy 
---------------------------------------------
resnet50             | 99.20%
efficientnet_b0      | 99.50%
vit                  | 96.80%


In [153]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 1. Define Path
# Targeting the root 'versions/1' to catch all subfolders (120, 140, 152, etc.)
dalle_root = external_data_map.get("DALLE_3", "")

print(f"Targeting DALL-E 3 Dataset Root: {dalle_root}")

# 2. Define Dataset Class (Pure Fake/AI)
class Dalle3EvalDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Recursively find all images in all subfolders
        valid_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')
        
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(valid_exts):
                    self.image_paths.append(os.path.join(root, file))
                    # Label 1 for FAKE (since this entire dataset is DALL-E 3)
                    self.labels.append(1)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            # FIX: Convert to RGBA -> RGB to handle transparency/GIFs safely
            image = Image.open(img_path).convert("RGBA").convert("RGB")
            
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros((3, 224, 224)), 1

# 3. Setup DataLoader
# Reusing the standard eval_transform
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dalle_dataset = Dalle3EvalDataset(dalle_root, transform=eval_transform)
dalle_loader = DataLoader(dalle_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"DALL-E 3 Dataset Loaded. Total Images: {len(dalle_dataset)}")
print("Note: All images are labeled as 'Fake' (1).")

# 4. Run Benchmark
# (Assuming 'results' dict and 'benchmark_model' function exist from previous cells)

print(f"\n{'Model':<20} | {'DALL-E 3 Detection Rate (Recall)':<35}")
print("-" * 60)

for model_name, data in results.items():
    if data['model']:
        # This function returns "accuracy", which in this case = Recall (True Positives / Total Fakes)
        acc = benchmark_model(data['model'], dalle_loader, model_name)
        print(f"{model_name:<20} | {acc:.2f}%")
    else:
        print(f"{model_name:<20} | Model not loaded")

Targeting DALL-E 3 Dataset Root: /Users/souhardyasaha/.cache/kagglehub/datasets/sourceduty/chatgpt-dall-e-3-images-and-sliced-gifs/versions/3
DALL-E 3 Dataset Loaded. Total Images: 412
Note: All images are labeled as 'Fake' (1).

Model                | DALL-E 3 Detection Rate (Recall)   
------------------------------------------------------------
resnet50             | 100.00%
efficientnet_b0      | 100.00%
vit                  | 99.76%
